# 📖 Karpathy's LLM Wiki 패턴 실습

이번 실습에서는 Andrej Karpathy가 제안한 **LLM Wiki** 패턴을 구현합니다.  
기존 RAG(벡터 검색)과 달리, LLM이 문서를 직접 읽고 **구조화된 위키 페이지**를 작성한 뒤  
해당 위키를 컨텍스트로 사용해 질문에 답하는 방식입니다.

> **원본 아이디어**: [Karpathy's GitHub Gist](https://gist.github.com/karpathy/442a6bf555914893e9891c11519de94f)

---

## RAG vs LLM Wiki 비교

| 구분 | 기존 RAG | LLM Wiki |
|------|----------|----------|
| **전처리** | 청크 분할 → 임베딩 → 벡터 DB | LLM이 문서 읽고 위키 페이지 직접 작성 |
| **검색** | 벡터 유사도 검색 | 컴파일된 위키를 컨텍스트에 직접 로드 |
| **지식 형태** | 원본 텍스트 청크 그대로 | 요약·크로스링크·백과사전 형식으로 재구성 |
| **유지보수** | 없음 | LLM이 주기적으로 헬스체크 및 보완 |
| **적합한 규모** | 대규모 문서 | 수백~수천 개 문서의 특화 도메인 |

---

## 📋 목차
1. [환경 설정](#1-환경-설정)
2. [STEP 1 - Ingest: 문서 로딩](#2-step-1---ingest-문서-로딩)
3. [STEP 2 - Compile: 위키 페이지 생성](#3-step-2---compile-위키-페이지-생성)
4. [STEP 3 - Query: 위키 기반 질문 응답](#4-step-3---query-위키-기반-질문-응답)
5. [STEP 4 - Maintain: 위키 헬스체크](#5-step-4---maintain-위키-헬스체크)
6. [RAG vs LLM Wiki 비교 실험](#6-rag-vs-llm-wiki-비교-실험)

---
## 1. 환경 설정

In [1]:
# 필요한 패키지 설치
# %pip install -qU langchain langchain-openai langchain-community pypdf python-dotenv

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if OPENROUTER_API_KEY:
    print("✅ API 키가 설정되었습니다.")
else:
    print("❌ API 키가 설정되지 않았습니다. .env 파일을 확인해주세요.")

✅ API 키가 설정되었습니다.


In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model           = "openai/gpt-oss-120b:free",
    openai_api_key  = OPENROUTER_API_KEY,
    openai_api_base = "https://openrouter.ai/api/v1",
)

print("✅ LLM 초기화 완료")

✅ LLM 초기화 완료


---
## 2. STEP 1 - Ingest: 문서 로딩

LLM Wiki의 첫 번째 단계는 원본 문서를 로드하고 `raw/` 폴더에 Markdown 형식으로 저장하는 것입니다.  
기존 RAG와 동일하게 PDF를 파싱하지만, **청크 분할이나 임베딩은 하지 않습니다.**

In [4]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "docs/DeepSeek_OCR_paper.pdf"

loader = PyPDFLoader(file_path)
docs = loader.load()

print(f"📄 로드된 페이지 수: {len(docs)}")

📄 로드된 페이지 수: 22


In [5]:
# raw/ 폴더에 전체 텍스트 저장
WIKI_DIR = Path("wiki")
RAW_DIR  = WIKI_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# 전체 문서를 하나의 raw 파일로 저장
raw_text = "\n\n".join(
    f"=== Page {doc.metadata.get('page', i) + 1} ===\n{doc.page_content}"
    for i, doc in enumerate(docs)
)

raw_file = RAW_DIR / "DeepSeek_OCR_paper.md"
raw_file.write_text(raw_text, encoding="utf-8")

print(f"✅ raw 파일 저장 완료: {raw_file}")
print(f"📊 전체 텍스트 길이: {len(raw_text):,} 문자")

✅ raw 파일 저장 완료: wiki\raw\DeepSeek_OCR_paper.md
📊 전체 텍스트 길이: 53,622 문자


---
## 3. STEP 2 - Compile: 위키 페이지 생성

LLM Wiki의 핵심 단계입니다.  
LLM이 raw 문서를 읽고 **토픽별 구조화된 위키 페이지**를 직접 작성합니다.

### 위키 페이지 구성
- `overview.md` — 논문 개요 및 핵심 기여
- `architecture.md` — 모델 아키텍처 (DeepEncoder, 디코더)
- `training.md` — 학습 방법 및 데이터
- `benchmarks.md` — 성능 및 벤치마크
- `limitations.md` — 한계점 및 향후 연구 방향

> 💡 **포인트**: 각 페이지는 `[[PageName]]` 문법으로 다른 페이지를 참조합니다.  
> 이것이 단순 요약과 달리 **지식이 연결되는 위키**의 특징입니다.

In [6]:
WIKI_PAGES = {
    "overview"     : "개요 (Overview)",
    "architecture" : "아키텍처 (Architecture)",
    "training"     : "학습 방법 (Training)",
    "benchmarks"   : "성능 및 벤치마크 (Benchmarks)",
    "limitations"  : "한계 및 향후 연구 (Limitations & Future Work)",
}

def compile_wiki_page(topic_key: str, topic_name: str, raw_content: str) -> str:
    """LLM을 사용해 특정 토픽의 위키 페이지를 생성합니다."""
    
    other_pages = [name for key, name in WIKI_PAGES.items() if key != topic_key]
    cross_links = ", ".join(f"[[{name}]]" for name in other_pages)

    prompt = f"""당신은 기술 위키 작성 전문가입니다.
아래 논문 원문을 바탕으로 \"{topic_name}\" 토픽의 위키 페이지를 한국어로 작성하세요.

## 작성 규칙
- Markdown 형식 사용 (##, ###, 불릿 포인트)
- 구체적인 수치와 기술적 세부사항 포함
- 관련 있는 다른 위키 페이지는 [[페이지명]] 형식으로 크로스링크 작성
  (연결 가능한 페이지: {cross_links})
- 마지막에 "## 관련 항목" 섹션 추가
- 사실에 기반하여 작성, 논문에 없는 내용은 추가하지 말 것

## 논문 원문
{raw_content}

## \"{topic_name}\" 위키 페이지:"""

    response = llm.invoke([{"role": "user", "content": prompt}])
    return response.content


print("✅ 위키 컴파일 함수 정의 완료")
print(f"📚 생성할 위키 페이지: {list(WIKI_PAGES.values())}")

✅ 위키 컴파일 함수 정의 완료
📚 생성할 위키 페이지: ['개요 (Overview)', '아키텍처 (Architecture)', '학습 방법 (Training)', '성능 및 벤치마크 (Benchmarks)', '한계 및 향후 연구 (Limitations & Future Work)']


In [7]:
# 위키 페이지 컴파일 실행
# ⚠️ LLM 호출이 5회 발생합니다 (페이지당 1회)

wiki_pages: dict[str, str] = {}

for key, name in WIKI_PAGES.items():
    print(f"⏳ 컴파일 중: {name} ...", end=" ", flush=True)
    wiki_pages[key] = compile_wiki_page(key, name, raw_text)
    
    # 파일로 저장
    output_file = WIKI_DIR / f"{key}.md"
    output_file.write_text(wiki_pages[key], encoding="utf-8")
    print(f"✅ 저장 완료 ({len(wiki_pages[key]):,} 문자)")

print("\n🎉 모든 위키 페이지 컴파일 완료!")
print(f"📂 저장 위치: {WIKI_DIR.resolve()}")

⏳ 컴파일 중: 개요 (Overview) ... ✅ 저장 완료 (2,568 문자)
⏳ 컴파일 중: 아키텍처 (Architecture) ... ✅ 저장 완료 (4,046 문자)
⏳ 컴파일 중: 학습 방법 (Training) ... ✅ 저장 완료 (3,135 문자)
⏳ 컴파일 중: 성능 및 벤치마크 (Benchmarks) ... ✅ 저장 완료 (4,693 문자)
⏳ 컴파일 중: 한계 및 향후 연구 (Limitations & Future Work) ... ✅ 저장 완료 (3,081 문자)

🎉 모든 위키 페이지 컴파일 완료!
📂 저장 위치: C:\Users\rkfka\workspace\FC_Agent_bible\Part 1. AI 에이전트 기초 다지기\CH03.LLM 어플리케이션의 기본, RAG\wiki


In [8]:
# 생성된 위키 페이지 미리보기
print("=" * 60)
print("📖 개요 (overview.md) 미리보기")
print("=" * 60)
print(wiki_pages["overview"][:1000])
print("...")

📖 개요 (overview.md) 미리보기
## 개요 (Overview)

**DeepSeek‑OCR**은 장문 텍스트를 **광학 2D 매핑**을 통해 압축하고, 압축된 비전 토큰을 기반으로 OCR(Optical Character Recognition) 디코딩을 수행하는 최초 연구 중 하나이다. 모델은 크게 두 부분으로 구성된다.

| 구성 요소 | 역할 | 파라미터 |
|-----------|------|----------|
| **DeepEncoder** | 고해상도 입력을 저활성화 메모리와 최소 비전 토큰 수로 처리·압축 | ≈ 380 M (80 M SAM‑base + 300 M CLIP‑large) |
| **DeepSeek‑3B‑MoE‑A570M** (Decoder) | 압축된 비전 토큰을 텍스트 토큰으로 복원 | 3 B 전체 파라미터, 추론 시 ≈ 570 M 활성 파라미터 (6/64 라우팅 + 2 공유 expert) |

### 핵심 아이디어
- **시각 토큰 기반 텍스트 압축**  
  하나의 이미지에 문서 텍스트 전체를 시각화하면, 디지털 텍스트와 비교해 훨씬 적은 토큰으로 정보를 표현할 수 있다.  
- **압축‑복원 매핑**  
  DeepEncoder가 **window‑attention** 기반 SAM와 **global‑attention** 기반 CLIP 사이에 16× 컨볼루션 토큰 압축기(2‑layer, kernel 3, stride 2, padding 1)를 두어, 초고해상도 입력을 1/16 토큰 수로 감소시킨 뒤 CLIP에 전달한다.  
- **다중 해상도 지원**  
  Tiny‑Small‑Base‑Large‑Gundam‑Gundam‑M 등 6가지 모드(해상도 512~1280 px, 토큰 64~~800)로 하나의 모델이 다양한 입력 규모에 대응한다.  

### 압축 효율 및 정확도
- **텍스트 : 비전 토큰 비율 10× 이하**(예: 64 vision tokens) → OCR 정밀도 **96 % ~ 98 %** (Fox bench

---
## 4. STEP 3 - Query: 위키 기반 질문 응답

컴파일된 위키 페이지를 직접 컨텍스트에 로드하여 질문에 답합니다.  
**벡터 검색 없이** 위키 전체(또는 관련 페이지)를 프롬프트에 포함시킵니다.

> 💡 소규모 도메인 특화 지식베이스에서는 이 방식이 더 일관성 있는 답변을 줍니다.

In [9]:
def load_wiki_context() -> str:
    """저장된 위키 파일들을 모두 읽어 하나의 컨텍스트 문자열로 반환합니다."""
    context_parts = []
    for key, name in WIKI_PAGES.items():
        wiki_file = WIKI_DIR / f"{key}.md"
        content = wiki_file.read_text(encoding="utf-8")
        context_parts.append(f"# [{name}]\n\n{content}")
    return "\n\n" + "="*50 + "\n\n".join(context_parts)


def query_wiki(question: str) -> str:
    """위키 컨텍스트를 기반으로 질문에 답합니다."""
    wiki_context = load_wiki_context()

    system_prompt = f"""당신은 지식 위키를 기반으로 질문에 답하는 어시스턴트입니다.
아래 위키 지식베이스를 참고하여 정확하고 구체적으로 답변하세요.
위키에 없는 내용은 '위키에 해당 정보가 없습니다'라고 답하세요.

## 위키 지식베이스
{wiki_context}"""

    response = llm.invoke([
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": question},
    ])
    return response.content


print("✅ 위키 쿼리 함수 정의 완료")

✅ 위키 쿼리 함수 정의 완료


In [10]:
# 테스트 1: 단순 사실 질문
q1 = "DeepSeek-OCR의 압축률(compression ratio)은 얼마인가요?"
print(f"❓ 질문: {q1}\n")
print(f"💡 답변:\n{query_wiki(q1)}")

❓ 질문: DeepSeek-OCR의 압축률(compression ratio)은 얼마인가요?

💡 답변:
DeepSeek‑OCR의 압축률은 **텍스트 토큰 대비 Vision 토큰 비율이 대략 10 × ~ 20 ×** 입니다. 구체적으로는 64 ~ 100 개의 Vision 토큰을 사용했을 때 텍스트 토큰 수(600 ~ 1300) 대비 **10.5× ~ 19.7×** 정도의 압축이 이루어집니다. 이는 논문·위키에 제시된 주요 수치이며, 압축 비율이 높아질수록 OCR 정확도가 약간 감소하는 경향이 있습니다.


In [11]:
# 테스트 2: 여러 위키 페이지를 종합해야 하는 복합 질문
q2 = "DeepSeek-OCR의 아키텍처 구성 요소와 각 컴포넌트의 학습 방법을 함께 설명해주세요."
print(f"❓ 질문: {q2}\n")
print(f"💡 답변:\n{query_wiki(q2)}")

❓ 질문: DeepSeek-OCR의 아키텍처 구성 요소와 각 컴포넌트의 학습 방법을 함께 설명해주세요.

💡 답변:
**DeepSeek‑OCR**는 Encoder‑Decoder 구조를 갖는 엔드‑투‑엔드 VLM이며, 크게 **DeepEncoder**와 **DeepSeek‑3B‑MoE‑A570M(Decoder)** 로 이루어집니다. 아래에서는 각 컴포넌트의 역할·구성·학습 방식을 한 번에 정리합니다.

---

## 1️⃣ 아키텍처 구성 요소

| 컴포넌트 | 핵심 역할 | 주요 서브 모듈(파라미터) | 입력·출력 |
|----------|----------|--------------------------|-----------|
| **DeepEncoder** | 고해상도 이미지를 **저활성화 메모리**와 **극소량 Vision Token** 으로 압축 | *SAM‑base* (80 M, patch‑size 16, 윈도우‑Attention) <br> *16× 토큰 압축기* (2‑layer Conv, kernel 3, stride 2, padding 1, 채널 256→1024) <br> *CLIP‑large* (300 M, patch‑size 16, 전역‑Dense Attention, 첫 patch‑embedding 제거) | `이미지 (512–1280 px)` → `압축된 Vision Token (64~800 개)` |
| **DeepSeek‑3B‑MoE‑A570M Decoder** | 압축된 Vision Token을 **텍스트 토큰**(OCR 결과) 으로 복원 | MoE 구조: 64 experts / 6 routed + 2 shared → **활성 파라미터 ≈ 570 M** (전체 3 B) | `Vision Token (N≤vision‑tokens)` → `텍스트 시퀀스 (OCR 문자열)` |
| **Prompt‑Control** (암묵적) | 하나의 모델이 OCR·차트·화학식·기하학 등 다양한 태스크를 수행하도록 **프롬프트**를 앞에 삽입 | – | 예: 

In [12]:
# 테스트 3: 한계점 및 실용적 시사점
q3 = "DeepSeek-OCR를 실제 서비스에 도입할 때 고려해야 할 한계점은 무엇인가요?"
print(f"❓ 질문: {q3}\n")
print(f"💡 답변:\n{query_wiki(q3)}")

❓ 질문: DeepSeek-OCR를 실제 서비스에 도입할 때 고려해야 할 한계점은 무엇인가요?

💡 답변:
**DeepSeek‑OCR 를 실제 서비스에 적용할 때 유의해야 할 주요 한계점**  

| 구분 | 구체적인 한계 | 서비스 적용 시 고려 사항 |
|------|--------------|--------------------------|
| **압축‑정밀도 트레이드‑오프** | 10 × 이하(≈ 64 ~ 100 vision tokens)에서는 OCR 정확도가 96 % ~ 98 % 수준이지만, 압축 비율이 10 × ~ 20 × 로 커질수록 정확도가 급격히 떨어져 20 × 압축 시 약 60 % 수준에 머뭅니다. | • 서비스 요구 정확도에 따라 **최대 압축 비율**을 10 × 이하로 제한.<br>• 고정밀이 필요한 문서(법률·의료·금융 등)에서는 **Tiny/Small 모드(64~100 tokens)** 를 사용하고, 저정밀 장면(대량 라벨링·예비 스캔)에서는 20 × 압축을 허용. |
| **긴 문서·복잡 레이아웃** | 텍스트 토큰 수가 4 k ~ 5 k 이상인 신문·학술 논문 등에서는 10 × 압축 한계를 초과하면 성능이 크게 저하됩니다. | • **Gundam / Gundam‑M** 모드(타일링 + 글로벌 뷰)와 같이 **다중 해상도**를 적용해 페이지를 여러 타일로 나누어 처리하도록 설계.<br>• 페이지당 토큰 수가 1 k 초과되는 경우 **분할 처리**(예: 페이지를 2~3개 청크로 나눔) 를 사전에 수행. |
| **해상도·블러링** | 512 × 512 또는 640 × 640 으로 다운샘플링된 이미지에서는 작은 글씨가 흐려져 인식률이 감소합니다. | • 입력 이미지 최소 **1024 × 1024**(Base 모드) 이상을 유지하거나, 중요한 영역에 대해 **선택적 고해상도 재압축**(zoom‑in) 전략을 적용.<br>• 전처리 단계에서 **슈퍼‑해상도** 또는 **노이즈 감소** 필터를 삽입해 품질 보강. |
| **다중 언어·소수 언

---
## 5. STEP 4 - Maintain: 위키 헬스체크

LLM Wiki의 차별화 기능입니다.  
LLM이 위키 전체를 검토하고 **누락된 정보, 비일관성, 보완할 크로스링크**를 찾아냅니다.

> 💡 **Karpathy의 핵심 아이디어**:  
> *"Obsidian이 IDE라면, LLM이 프로그래머이고, 위키가 코드베이스다."*  
> 인간은 소스를 제공하고 질문만 하면, LLM이 지식 구조를 유지·발전시킵니다.

In [13]:
def health_check_wiki() -> str:
    """위키 전체를 검토하여 개선점을 찾아냅니다."""
    wiki_context = load_wiki_context()

    prompt = f"""당신은 위키 유지보수 전문가입니다.
아래 위키 지식베이스를 전체적으로 검토하고 헬스체크 리포트를 작성하세요.

## 검토 항목
1. **비일관성**: 페이지 간 서로 상충되거나 모순된 내용
2. **누락된 정보**: 중요하지만 위키에 없는 내용
3. **크로스링크 기회**: 연결되어야 하지만 아직 연결되지 않은 개념들
4. **보완 우선순위**: 가장 시급하게 보완해야 할 페이지와 이유

## 위키 지식베이스
{wiki_context}

## 헬스체크 리포트 (한국어로 작성):"""

    response = llm.invoke([{"role": "user", "content": prompt}])
    return response.content


print("⏳ 위키 헬스체크 실행 중...\n")
report = health_check_wiki()
print(report)

⏳ 위키 헬스체크 실행 중...

# 🛠️ DeepSeek‑OCR 위키 헬스체크 리포트  
*(2026‑05‑10 기준)*  

---

## 1. 비일관성 (Inconsistencies)

| 페이지 | 충돌·모순 내용 | 원인 추정 | 해결 권고 |
|--------|----------------|----------|-----------|
| **개요** ↔ **아키텍처** | • 개요에서는 “16× 토큰 압축기”라고 기술했으나, 아키텍처 표에서는 “16× 토큰 압축기” → 실제 연산은 **4‑step**(stride 2 × 2) → 1/16 압축. <br>• 개요에선 “6/64 라우팅 + 2 공유 expert”라고 적히지만, 아키텍처 섹션에선 “6/64 라우팅, 2 공유 expert”가 **활성 파라미터 570 M**이라는 설명이 중복돼 혼동. | 표와 서술이 동일하게 정리되지 않음. | 두 페이지에 동일한 표·문장을 삽입하고, “압축기”와 “MoE 라우팅” 설명을 하나의 정의(예: **“16× 압축기 = 2‑layer Conv, stride 2 → 1/16 토큰”**, **“MoE 라우팅: 6 active experts / 64 total + 2 shared”**) 로 통일. |
| **성능 및 벤치마크** ↔ **학습 방법** | • Benchmarks 테이블에 “텍스트 : 비전 10× 이하 → 96 %~98 %”라고 적히나, 학습 방법에서는 “10× 압축 시 96.5 %”만 제시하고 100 vision tokens(6.7×)에 대한 수치는 누락. | 실험 결과가 페이지 간에 분산돼 전체 그림을 파악하기 어려움. | Benchmarks 페이지에 **“학습 방법 섹션 5‑표와 동일”**이라는 주석을 달고, 학습 방법에선 **“표 1 (Benchmarks) 참조”** 라고 명시. |
| **한계** ↔ **개요** | 한계에서는 “압축 비율 20×에서 60 % 이하 정확도”라고 명시하지만, 개요에서는 “20× 압축 → 60 % ~ *98 %*”이라고 불명

---
## 6. RAG vs LLM Wiki 비교 실험

동일한 질문에 대해 기존 RAG(벡터 검색)와 LLM Wiki의 답변을 비교합니다.

In [14]:
# 비교를 위해 기존 RAG 파이프라인 구성
import httpx
from langchain_core.embeddings import Embeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

class OpenRouterEmbeddings(Embeddings):
    def __init__(self, model: str, api_key: str):
        self.model   = model
        self.api_key = api_key
        self.headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type":  "application/json",
        }

    def _embed(self, texts: list[str]) -> list[list[float]]:
        resp = httpx.post(
            "https://openrouter.ai/api/v1/embeddings",
            headers=self.headers,
            json={"model": self.model, "input": texts},
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()["data"]
        return [item["embedding"] for item in sorted(data, key=lambda x: x["index"])]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self._embed(texts)

    def embed_query(self, text: str) -> list[float]:
        return self._embed([text])[0]


# 텍스트 분할 및 벡터 DB 구성
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits   = splitter.split_documents(docs)

embeddings   = OpenRouterEmbeddings(
    model   = "nvidia/llama-nemotron-embed-vl-1b-v2:free",
    api_key = OPENROUTER_API_KEY,
)
vector_store = InMemoryVectorStore.from_documents(splits, embedding=embeddings)
retriever    = vector_store.as_retriever(search_kwargs={"k": 4})

print(f"✅ RAG 파이프라인 준비 완료 ({len(splits)}개 청크)")

✅ RAG 파이프라인 준비 완료 (73개 청크)


In [15]:
def query_rag(question: str) -> str:
    """기존 RAG 방식으로 질문에 답합니다."""
    retrieved = retriever.invoke(question)
    context   = "\n\n".join(doc.page_content for doc in retrieved)

    response = llm.invoke([
        {"role": "system", "content": f"다음 문서를 참고하여 질문에 답하세요.\n\n{context}"},
        {"role": "user",   "content": question},
    ])
    return response.content


def compare(question: str):
    """동일 질문에 대해 RAG와 LLM Wiki 답변을 나란히 출력합니다."""
    print(f"❓ 질문: {question}")
    print("\n" + "─" * 60)

    print("\n🔍 [기존 RAG 답변]")
    print(query_rag(question))

    print("\n" + "─" * 60)
    print("\n📖 [LLM Wiki 답변]")
    print(query_wiki(question))
    print("\n" + "=" * 60)


print("✅ 비교 함수 정의 완료")

✅ 비교 함수 정의 완료


In [16]:
# 비교 실험 1: 단순 사실 질문
compare("DeepSeek-OCR의 압축률(compression ratio)은 얼마인가요?")

❓ 질문: DeepSeek-OCR의 압축률(compression ratio)은 얼마인가요?

────────────────────────────────────────────────────────────

🔍 [기존 RAG 답변]
DeepSeek‑OCR는 **텍스트 → 비전 토큰** 사이의 압축 비율을 조절해가며 실험을 진행했으며, 논문에서 보고된 주요 압축률은 다음과 같습니다.

| 압축률 (Compression Ratio) | 텍스트 대비 비전 토큰 수 | OCR 정확도(정밀도) |
|----------------------------|--------------------------|-------------------|
| **≈ 9–10×**                | 텍스트 토큰이 비전 토큰의 9~10배 | **96 % 이상** |
| **≈ 10–12×**               | 텍스트 토큰이 비전 토큰의 10~12배 | **≈ 90 %** |
| **≈ 20×**                  | 텍스트 토큰이 비전 토큰의 20배 | **≈ 60 %** |

즉, **DeepSeek‑OCR가 실현할 수 있는 압축률은 최소 9× 정도부터 최대 20× 정도까지**이며, 압축률이 낮을수록 (예: 9–10×) OCR 정밀도가 96 % 이상으로 매우 높게 유지됩니다. 압축률이 20×에 달하면 정확도가 약 60 % 수준으로 떨어지지만, 여전히 실용적인 성능을 보여줍니다.  

논문에서는 이 범위(≈ 9× ~ 20×)를 “텍스트 토큰 수가 비전 토큰 수의 10배 이하(압축 비율 < 10×)일 때 97 % 수준의 정밀도”, “압축 비율 20×에서도 약 60 % 정밀도”라는 형태로 강조하고 있습니다. 따라서 DeepSeek‑OCR의 **핵심 압축률**은 **9× ~ 20×** 구간이라고 보시면 됩니다.

────────────────────────────────────────────────────────────

📖 [LLM Wiki 답변]
DeepSeek‑

In [17]:
# 비교 실험 2: 여러 섹션을 종합해야 하는 질문
compare("DeepSeek-OCR와 경쟁 모델들(GOT-OCR2.0, MinerU2.0)을 비교하고 한계점도 설명해주세요.")

❓ 질문: DeepSeek-OCR와 경쟁 모델들(GOT-OCR2.0, MinerU2.0)을 비교하고 한계점도 설명해주세요.

────────────────────────────────────────────────────────────

🔍 [기존 RAG 답변]
## 1.  경쟁 모델 개요  

| 모델 | 주요 특징 | 사용된 비전 토큰(평균) | 입력 해상도 | 대표 성능 지표 (OmniDocBench edit‑distance) |
|------|-----------|------------------------|------------|---------------------------------------------|
| **DeepSeek‑OCR** | LLM‑VLM 하이브리드, 토큰‑압축 설계, 3가지 모드 (Small / Base / Large + Gundam) | 100 ≈ 400 ≈ < 800 (모드에 따라) | 640 × 640 (Small) → 1280 × 1280 (Gundam) | **0.037 ~ 0.156** (전체 영역, 영어/중국어 모두) |
| **GOT‑OCR 2.0** | “Vision‑token → LLM” 파이프라인, 256 비전 토큰 사용 | 256 | 1280 × 1280 (보통) | 0.08 ~ 0.31 (표/수식 포함) |
| **MinerU 2.0** | 대용량 비전 토큰 + 멀티‑스케일 윈도우, 6 000 + 토큰/페이지 | ≈ 7 000 | 1500 × 1500 ~ 2000 × 2000 | 0.10 ~ 0.58 (다양한 레이아웃) |

> **Edit‑distance**가 작을수록 OCR 결과가 원본 텍스트와 가깝습니다. 표에 나타난 값은 같은 페이지당 평균값이며, DeepSeek‑OCR는 같은 혹은 더 좋은 정확도를 훨씬 적은 비전 토큰으로 달성합니다.

---

## 2.  DeepSeek‑OCR vs. 경쟁 모델  

| 비교 항목 | DeepSeek‑OCR | GOT‑OCR 2.0 | MinerU 2.0 |


---
## 📝 정리: LLM Wiki 패턴의 특징

### 장점
- **높은 응답 일관성**: 위키가 미리 구조화되어 있어 답변 품질이 안정적
- **크로스링크**: 개념 간 연결이 명시적으로 표현됨
- **유지보수 가능**: 헬스체크로 지식베이스를 지속적으로 개선
- **벡터 DB 불필요**: 소규모 도메인에서 인프라 단순화

### 한계
- **컨텍스트 윈도우 제한**: 위키가 커질수록 전체 로드 불가
- **초기 컴파일 비용**: LLM 호출이 페이지 수만큼 발생
- **대규모 문서 비적합**: 수만 개 이상의 문서에는 RAG가 더 효율적
- **LLM 환각 위험**: 컴파일 단계에서 LLM이 잘못된 내용을 작성할 수 있음

### 적합한 사용 사례
- 특정 도메인의 논문/문서 수백 개 수준
- 지식이 자주 업데이트되지 않는 경우
- 개념 간 관계가 중요한 질문이 많은 경우

---
### 📚 참고 자료
- [Karpathy's LLM Wiki Gist](https://gist.github.com/karpathy/442a6bf555914893e9891c11519de94f)
- [Beyond RAG: LLM Wiki Pattern - Level Up Coding](https://levelup.gitconnected.com/beyond-rag-how-andrej-karpathys-llm-wiki-pattern-builds-knowledge-that-actually-compounds-31a08528665e)